In [2]:
import pandas as pd
data = pd.read_csv("data.csv")

In [4]:
 data = data[data["prix_eur"]<5000000]

In [52]:
from sklearn.preprocessing import LabelEncoder

In [53]:
encoder_type = LabelEncoder()
data["type_bien"] = encoder_type.fit_transform(data["type_bien"])

In [54]:
encoder_etage = LabelEncoder()
data["etage"] = encoder_type.fit_transform(data["etage"])

In [55]:
encoder_adresse = LabelEncoder()
data["adresse"] = encoder_type.fit_transform(data["adresse"])

In [58]:
mapping_dpe = {"A": 1, "B": 2, "C": 3, "D": 4, "E": 5, "F": 6, "G": 7}
data["dpe"] = data["dpe"].map(mapping_dpe)


In [59]:
data

,Unnamed: 0,prix_eur,prix_m2,type_bien,surface_m2,nb_pieces,nb_chambres,etage,dpe,adresse,code_postal,url,latitude,longitude
0,0.0,630000.0,9960.0,4,63.3,3.0,2.0,26,4.0,87,75011.0,https://www.seloger.com/annonces/achat/apparte...,NaN,NaN
1,1.0,814000.0,7752.0,4,105.0,5.0,3.0,127,5.0,156,75020.0,https://www.seloger.com/annonces/achat/apparte...,NaN,NaN
2,2.0,725000.0,12719.0,4,57.0,3.0,2.0,165,5.0,91,75006.0,https://www.seloger.com/annonces/achat/apparte...,NaN,NaN
3,3.0,1090000.0,13293.0,4,82.0,3.0,2.0,223,6.0,35,75016.0,https://www.seloger.com/annonces/achat/apparte...,NaN,NaN
4,4.0,600000.0,9524.0,4,63.0,3.0,2.0,129,6.0,20,75012.0,https://www.seloger.com/annonces/achat/apparte...,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9866,26.0,549000.0,10558.0,4,52.0,2.0,1.0,26,5.0,67,75007.0,https://www.seloger.com/annonces/achat/apparte...,NaN,NaN
9867,27.0,1350000.0,14136.0,4,95.5,4.0,2.0,223,4.0,132,75008.0,https://www.seloger.com/annonces/achat/apparte...,NaN,NaN
9868,28.0,484000.0,12281.0,4,39.4,2.0,1.0,223,5.0,165,75006.0,https://www.seloger.com/annonces/achat/apparte...,NaN,NaN
9869,29.0,2020000.0,16694.0,4,121.0,4.0,2.0,208,5.0,131,75007.0,https://www.seloger.com/annonces/achat/apparte...,NaN,NaN


In [60]:
y= data['prix_eur']
y

0        630000.0
1        814000.0
2        725000.0
3       1090000.0
4        600000.0
          ...    
9866     549000.0
9867    1350000.0
9868     484000.0
9869    2020000.0
9870     200000.0
Name: prix_eur, Length: 9871, dtype: float64

In [61]:
X= data.drop(["Unnamed: 0", "url", "latitude", "longitude","prix_eur", "prix_m2"], axis=1)
X = X.fillna(-100000)

In [62]:
y_train= y[0:int(len(X)*0.8)]
y_test=y[int(len(X)*0.8):]
X_train=X[0:int(len(X)*0.8)]
X_test=X[int(len(X)*0.8):]

In [65]:

from sklearn.ensemble import GradientBoostingRegressor

model = GradientBoostingRegressor(n_estimators=300, learning_rate=0.05,
                                   max_depth=5, random_state=42)

model.fit(X_train,y_train)

,loss,'squared_error'
,learning_rate,0.05
,n_estimators,300
,subsample,1.0
,criterion,'friedman_mse'
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_depth,5
,min_impurity_decrease,0.0
,init,None


In [66]:
y_pred= model.predict(X_test)
metrics.mean_squared_error(y_test, y_pred)

1.112917500359223e+26

In [67]:
model.fit(X,y)

,loss,'squared_error'
,learning_rate,0.05
,n_estimators,300
,subsample,1.0
,criterion,'friedman_mse'
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_depth,5
,min_impurity_decrease,0.0
,init,None


In [74]:
import joblib

joblib.dump(encoder_etage, "utils/encoder_etage.joblib")
joblib.dump(encoder_adresse, "utils/encoder_adresse.joblib")
joblib.dump(encoder_type, "utils/encoder_type.joblib")
joblib.dump(model, "utils/model.joblib")


['utils/model.joblib']

In [ ]:
def ft_pred(infos_bien):
    
    encoder_etage = joblib.load("utils/encoder_etage.joblib")
    encoder_adresse = joblib.load("utils/encoder_adresse.joblib")
    encoder_type = joblib.load("utils/encoder_type.joblib")

    model = joblib.load("utils/model.joblib")

    mapping_dpe = {
        "A": 1, "B": 2, "C": 3,
        "D": 4, "E": 5, "F": 6, "G": 7
    }

    df = pd.DataFrame([{
        "type_bien": infos_bien["type_bien"],
        "etage": str(infos_bien["etage"]),
        "adresse": infos_bien["adresse"],
        "surface": infos_bien["surface"],
        "pieces": infos_bien["pieces"],
        "chambres": infos_bien["chambres"],
        "dpe": infos_bien["dpe"]
    }])

    df["type_bien"] = encoder_type.transform(df["type_bien"])
    df["etage"] = encoder_etage.transform(df["etage"])
    df["adresse"] = encoder_adresse.transform(df["adresse"])
    df["dpe"] = df["dpe"].map(mapping_dpe)

    df = df.fillna(-100000)

    estimation = model.predict(df)[0]

    return estimation

    
 